In [1]:
import subprocess
import concurrent.futures
from pathlib import Path
import os

In [ ]:
def run_prefetch(sra_list, ecotype_directory):
    """
    Downloads SRA files using `prefetch` for a given list of SRA accession numbers.

    Parameters:
    - sra_list (list): List of SRA accession numbers to download.
    - ecotype_directory (Path): Path to the directory where SRA files should be stored.
    """
    for sra_id in sra_list:
        prefetch_command = f'prefetch {sra_id}'
        try:
            subprocess.run(prefetch_command, shell=True,
                           check=True, cwd=ecotype_directory)
        except subprocess.CalledProcessError as e:
            print(
                f"Error running prefetch for {sra_id}. Exit code: {e.returncode}")


def move_sra_files(ecotype_directory):
    """
    Moves downloaded SRA files from subdirectories to the main ecotype directory.

    Parameters:
    - ecotype_directory (Path): Path where SRA files are stored.
    """
    move_command = 'mv *.sra ..'
    sra_subdirectories = list(ecotype_directory.glob("SRR*/"))

    for subdirectory in sra_subdirectories:
        try:
            subprocess.run(move_command, shell=True,
                           check=True, cwd=subdirectory)
        except subprocess.CalledProcessError:
            print(f"Error moving SRA files in {subdirectory}")


def generate_star_index(ecotype, ecotype_directory):
    """
    Generates a STAR genome index for alignment.

    Parameters:
    - ecotype (str): Name of the ecotype being processed.
    - ecotype_directory (Path): Path to the directory where STAR index will be stored.
    """
    star_index_command = f'STAR --runMode genomeGenerate \
                             --genomeDir {ecotype_directory} \
                             --genomeFastaFiles {ecotype_directory}/{ecotype}.fna \
                             --sjdbGTFfile {ecotype_directory}/{ecotype}.gtf \
                             --runThreadN 12'
    try:
        subprocess.run(star_index_command, shell=True,
                       check=True, cwd=ecotype_directory)
    except subprocess.CalledProcessError:
        print(f"Error generating STAR index for {ecotype}")


def process_fastq_files(ecotype, ecotype_directory):
    """
    Processes raw SRA files into FASTQ format using fasterq-dump.

    Parameters:
    - ecotype (str): Name of the ecotype being processed.
    - ecotype_directory (Path): Path where FASTQ files will be stored.
    """
    fastq_commands = [
        'fasterq-dump *.sra -e 5',
        f'cat *_1.fastq > {ecotype}_1.fastq',
        f'cat *_2.fastq > {ecotype}_2.fastq',
        'rm SRR*.fastq'
    ]

    for command in fastq_commands:
        try:
            subprocess.run(command, shell=True, check=True,
                           cwd=ecotype_directory)
        except subprocess.CalledProcessError:
            print(f"Error processing FASTQ files for {ecotype}")


def run_alignment_and_trimming(ecotype, ecotype_directory):
    """
    Runs quality trimming (Trimmomatic) and alignment (STAR).

    Parameters:
    - ecotype (str): Name of the ecotype being processed.
    - ecotype_directory (Path): Path where files will be stored.
    """
    trimming_command = f'trimmomatic PE -phred33 -threads 20 {ecotype}_1.fastq {ecotype}_2.fastq \
                         {ecotype}_1_trim.fastq {ecotype}_1_unpaired.fastq \
                         {ecotype}_2_trim.fastq {ecotype}_2_unpaired.fastq \
                         LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36 > {ecotype}_trim_log.txt'

    star_alignment_command = f'STAR --runThreadN 40 --genomeDir {ecotype_directory} \
                               --outFileNamePrefix {ecotype}. \
                               --readFilesIn {ecotype}_1_trim.fastq {ecotype}_2_trim.fastq \
                               --outSJfilterReads Unique --outFilterMismatchNmax 4 \
                               --quantMode GeneCounts --twopassMode Basic'

    cleanup_command = f'rm *.fastq'

    commands = [trimming_command, star_alignment_command, cleanup_command]

    for command in commands:
        try:
            subprocess.run(command, shell=True, check=True,
                           cwd=ecotype_directory)
        except subprocess.CalledProcessError:
            print(f"Error during alignment/trimming step for {ecotype}")


def run_as_analysis(ecotype, ecotype_directory):
    """
    Runs alternative splicing analysis using Perl scripts.

    Parameters:
    - ecotype (str): Name of the ecotype being processed.
    - ecotype_directory (Path): Path where AS analysis output will be stored.
    """
    perl_scripts = {
        "junction_count": f'perl /home/ASTool/junction_count.pl \
                            --gtf {ecotype_directory}/{ecotype}.gtf \
                            --sam {ecotype}.Aligned.out.sam \
                            --thread 40 --readlength 100 --m 8 --outdir {ecotype}_junction_ref',
        "intron_retention": f'perl /home/ASTool/IR_PSI.pl \
                              --junction {ecotype}_junction_ref/junction_count.txt \
                              --gtf {ecotype_directory}/{ecotype}.gtf --m 8 --outdir {ecotype}_IR_PSI.txt',
        "exon_skipping": f'perl /home/ASTool/ES_PSI.pl \
                           --junction {ecotype}_junction_ref/junction_count.txt \
                           --gtf {ecotype_directory}/{ecotype}.gtf --m 8 --outdir {ecotype}_ES_PSI.txt'
    }

    for step, command in perl_scripts.items():
        try:
            subprocess.run(command, shell=True, check=True,
                           cwd=ecotype_directory)
        except subprocess.CalledProcessError:
            print(f"Error in alternative splicing step ({step}) for {ecotype}")

In [ ]:
# Main execution
ecotypes = ['Arabidopsis_thaliana']
# Different RNA-seq data from different tissues of the species
sra_accessions = ['SRR18047059', 'SRR24745554', 'SRR6784793',
                  'SRR23313059', 'SRR23595725', 'SRR10914245', 'SRR23013330', 'SRR24283874']

for ecotype in ecotypes:
    ecotype_dir = Path(
        f"/data/Step1_training_data_preparation_index/{ecotype}")
    os.makedirs(ecotype_dir, exist_ok=True)
    # Prefetch SRA data in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        future = executor.submit(run_prefetch, sra_accessions, ecotype_dir)
        try:
            future.result()  # This will print exceptions inside threads
        except Exception as e:
            print(f"Error occurred during prefetching: {e}")
    move_sra_files(ecotype_dir)
    process_fastq_files(ecotype, ecotype_dir)
    generate_star_index(ecotype, ecotype_dir)
    run_alignment_and_trimming(ecotype, ecotype_dir)
    run_as_analysis(ecotype, ecotype_dir)